In [1]:
import re
import numpy as np
import nltk
from collections import defaultdict, Counter
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag

for resource in [
    "punkt",
    "punkt_tab",
    "wordnet",
    "omw-1.4",
    "averaged_perceptron_tagger_eng",
]:
    nltk.download(resource, quiet=True)


In [2]:
lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN


def lemmatize_words(words):
    tagged = pos_tag(words)
    return [lemmatizer.lemmatize(w, get_wordnet_pos(tag)) for w, tag in tagged]

In [ ]:
def preprocess_with_boundaries(text, use_lemmatization=True):
    text = re.sub(r'<[^>]+>', ' ', text)
    text = text.lower()
    sentences = sent_tokenize(text)

    flattened_tokens = []

    for sentence in sentences:
        sentence = re.sub(r'[^a-z\s]', ' ', sentence)
        sentence = re.sub(r'\s+', ' ', sentence).strip()

        if not sentence:
            continue

        words = word_tokenize(sentence)

        if use_lemmatization:
            words = lemmatize_words(words)

        if words:
            flattened_tokens.extend(['<s>'] + words + ['</s>'])

    return flattened_tokens


def load_corpus_tokens(use_lemmatization=True):
    filepath = "corpus.txt"

    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read()

    if not text.strip():
        raise ValueError("corpus.txt is empty. Put the corpus text in corpus.txt and keep it in the same folder as this notebook.")

    return preprocess_with_boundaries(text, use_lemmatization)


def build_ngram_model(tokens, n):
    counts = defaultdict(Counter)

    for i in range(len(tokens) - n + 1):
        history = tuple(tokens[i:i + n - 1])
        next_word = tokens[i + n - 1]
        counts[history][next_word] += 1

    prob_model = defaultdict(dict)

    for history, counter in counts.items():
        total = sum(counter.values())

        for word, count in counter.items():
            prob_model[history][word] = count / total

    return prob_model


def build_laplace_bigram_data(tokens):
    bigram_counts = defaultdict(Counter)
    history_counts = Counter()

    vocabulary = sorted(set(tokens) - {'<s>'})

    for i in range(len(tokens) - 1):
        history_word = tokens[i]
        next_word = tokens[i + 1]

        bigram_counts[history_word][next_word] += 1
        history_counts[history_word] += 1

    return {
        "bigram_counts": bigram_counts,
        "history_counts": history_counts,
        "vocabulary": vocabulary,
        "vocab_size": len(vocabulary),
    }


def laplace_bigram_predictor(history_word, laplace_data):
    bigram_counts = laplace_data["bigram_counts"]
    history_counts = laplace_data["history_counts"]
    vocabulary = laplace_data["vocabulary"]
    vocab_size = laplace_data["vocab_size"]

    if vocab_size == 0:
        return []

    history_count = history_counts.get(history_word, 0)
    denominator = history_count + vocab_size

    candidates = []

    for next_word in vocabulary:
        bigram_count = bigram_counts[history_word].get(next_word, 0)
        probability = (bigram_count + 1) / denominator
        candidates.append((next_word, probability))

    return sorted(candidates, key=lambda x: x[1], reverse=True)


def train_models():
    raw_tokens = load_corpus_tokens(use_lemmatization=False)
    lemma_tokens = load_corpus_tokens(use_lemmatization=True)

    raw_models = {
        4: build_ngram_model(raw_tokens, n=4),
        3: build_ngram_model(raw_tokens, n=3),
        2: build_ngram_model(raw_tokens, n=2),
        "laplace": build_laplace_bigram_data(raw_tokens),
    }

    lemma_models = {
        4: build_ngram_model(lemma_tokens, n=4),
        3: build_ngram_model(lemma_tokens, n=3),
        2: build_ngram_model(lemma_tokens, n=2),
        "laplace": build_laplace_bigram_data(lemma_tokens),
    }

    print(f"raw tokens: {len(raw_tokens)}, lemmatized tokens: {len(lemma_tokens)}")
    print(f"raw vocabulary: {raw_models['laplace']['vocab_size']}, lemmatized vocabulary: {lemma_models['laplace']['vocab_size']}")

    return {
        "raw": raw_models,
        "lemmatized": lemma_models,
    }


def backoff_predictor(history, models):
    """
    Backoff order:
        4-gram -> 3-gram -> ordinary seen bigram -> Laplace-smoothed bigram

    The final Laplace step prevents zero probability for an unseen history.
    Returns (candidates, unseen_history).
    """
    h4 = tuple(history[-3:])
    if len(h4) == 3 and h4 in models[4]:
        candidates = sorted(models[4][h4].items(), key=lambda x: x[1], reverse=True)
        return candidates, False

    h3 = tuple(history[-2:])
    if len(h3) == 2 and h3 in models[3]:
        candidates = sorted(models[3][h3].items(), key=lambda x: x[1], reverse=True)
        return candidates, False

    h2 = tuple(history[-1:])
    if len(h2) == 1 and h2 in models[2]:
        candidates = sorted(models[2][h2].items(), key=lambda x: x[1], reverse=True)
        return candidates, False

    if len(history) > 0:
        history_word = history[-1]
        unseen_history = models["laplace"]["history_counts"].get(history_word, 0) == 0
        candidates = laplace_bigram_predictor(history_word, models["laplace"])
        return candidates, unseen_history

    return [], False


def generate_text(seed_phrase, models, use_lemmatization=True, max_words=30):
    seed_clean = re.sub(r'[^a-z\s]', ' ', seed_phrase.lower())
    seed_clean = re.sub(r'\s+', ' ', seed_clean).strip()

    if seed_clean:
        seed_words = word_tokenize(seed_clean)
    else:
        seed_words = []

    if use_lemmatization:
        seed_words = lemmatize_words(seed_words)

    output_tokens = ['<s>'] + seed_words

    for _ in range(max_words):
        candidates, unseen_history = backoff_predictor(tuple(output_tokens), models)

        if not candidates:
            break

     
        if unseen_history:
            selected_candidates = candidates
        else:
            selected_candidates = candidates[:5]

        words, probs = zip(*selected_candidates)
        probs = np.array(probs, dtype=float)
        probs = probs / probs.sum()

        next_word = np.random.choice(words, p=probs)

        if next_word == '</s>':
            break

        output_tokens.append(next_word)

    generated = output_tokens[1:]
    return ' '.join(generated)


In [4]:
def run_generator():
    models = train_models()

    seed_phrase = input("Enter seed phrase (or press Enter for empty): ")

    raw_output = generate_text(seed_phrase, models["raw"], use_lemmatization=False)
    lemma_output = generate_text(seed_phrase, models["lemmatized"], use_lemmatization=True)

    print(f"\n[raw]        {raw_output}")
    print(f"[lemmatized] {lemma_output}")


In [5]:
if __name__ == '__main__':
    run_generator()

raw tokens: 40967, lemmatized tokens: 40967
raw vocabulary: 4598, lemmatized vocabulary: 3757

[raw]        it is true that you have been fortunate enough to gain he bowed me out of the hall looking back into the street
[lemmatized] he be a mr john turner who make his money in australia and return some year ago to the old country
